# Lab 5 — Pareto Analysis, Visualisation, and Convergence Metrics

## Background

Having found a Pareto-approximate set in Lab 4, two questions remain:

1. **Has the algorithm converged?** Are there still unexplored regions of the objective space?
2. **How much does the result depend on the random seed?** MOEAs are stochastic — different seeds
   give different results. A reliable analysis requires multiple seeds.

### Convergence metrics

Five metrics are commonly used to assess MOEA convergence. All require a **reference set**
(the best-known Pareto front, typically the final archive or a cross-seed union):

| Metric | What it measures | Desired trend |
|--------|-----------------|---------------|
| **Hypervolume** | Volume of objective space dominated by the Pareto set | Increasing → plateau |
| **ε-progress** | Cumulative count of new ε-grid cells discovered | Increasing → plateau |
| **Generational distance (GD)** | Average distance from archive to reference set | Decreasing → 0 |
| **ε-indicator** | Minimum ε such that archive ε-dominates reference set | Decreasing → 0 |
| **Inverted GD (IGD)** | Average distance from reference set to archive | Decreasing → 0 |
| **Spacing** | Standard deviation of nearest-neighbour distances in archive | Decreasing → 0 |

### Seed analysis

MOEAs use random crossover and mutation, so results vary between runs. Best practice:
1. Run the algorithm with 5 different random seeds.
2. Combine the final archives into a **cross-seed reference set** (via ε-nondominated sort).
3. Compute convergence metrics for each seed against the shared reference set.

Consistent convergence across seeds (lines collapse to same curve) indicates reliable results.

### What you will do

1. Load archives from Lab 4 and compute all convergence metrics.
2. Visualise the full metric suite for the 5k and 100k runs.
3. Run 5 random seeds and compare their Pareto fronts.
4. Build a cross-seed reference set with ε-nondominated sorting.
5. Compute per-seed convergence metrics and visualise.

## 1. Setup — model, imports, and helper functions

We define `calculate_metrics` and `plot_metrics` here. These functions are reused throughout
the lab.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from lakemodel_function import lake_problem
from ema_workbench import (Model, RealParameter, ScalarOutcome,
                           MultiprocessingEvaluator, Constant, ema_logging,
                           HypervolumeMetric, GenerationalDistanceMetric,
                           EpsilonIndicatorMetric, InvertedGenerationalDistanceMetric,
                           SpacingMetric)
from ema_workbench.analysis import parcoords
from ema_workbench import load_archives
from ema_workbench.em_framework.optimization import (
    epsilon_nondominated, Problem
)
from ema_workbench import Constraint

ema_logging.log_to_stderr(ema_logging.INFO)

# --- Rebuild model (same configuration as Lab 4) ---
lake_model = Model('lakeproblem', function=lake_problem)
lake_model.time_horizon = 100

lake_model.uncertainties = [
    RealParameter('mean',  0.01,  0.05),
    RealParameter('stdev', 0.001, 0.005),
    RealParameter('b',     0.1,   0.45),
    RealParameter('q',     2.0,   4.5),
    RealParameter('delta', 0.93,  0.99),
]
lake_model.levers = [RealParameter(f'l{i}', 0, 0.1)
                     for i in range(lake_model.time_horizon)]
lake_model.outcomes = [
    ScalarOutcome('max_P',       kind=ScalarOutcome.MINIMIZE),
    ScalarOutcome('utility',     kind=ScalarOutcome.MAXIMIZE),
    ScalarOutcome('inertia',     kind=ScalarOutcome.MAXIMIZE),
    ScalarOutcome('reliability', kind=ScalarOutcome.MAXIMIZE),
]
lake_model.constants = [Constant('alpha', 0.41), Constant('nsamples', 150)]

constraint_maxP = Constraint('max_pollution', outcome_names='max_P',
                              function=lambda x: max(0, x - 5))

In [ ]:
def calculate_metrics(archives, reference_set):
    """Compute 5 convergence metrics for every archive snapshot.
    
    Parameters
    ----------
    archives : list of (nfe, DataFrame) tuples
        Loaded by load_archives().
    reference_set : DataFrame
        The reference Pareto set to compare against.
    
    Returns
    -------
    pd.DataFrame with columns: nfe, hypervolume, generational_distance,
        epsilon_indicator, inverted_gd, spacing.
    """
    problem = Problem('levers', lake_model.levers, [o for o in lake_model.outcomes if o.kind != o.INFO])

    hv = HypervolumeMetric(reference_set, problem)
    gd = GenerationalDistanceMetric(reference_set, problem, d=1)
    ei = EpsilonIndicatorMetric(reference_set, problem)
    ig = InvertedGenerationalDistanceMetric(reference_set, problem, d=1)
    sm = SpacingMetric(reference_set, problem)

    metrics = []
    for nfe, archive in archives:
        metrics.append({
            'nfe':                  int(nfe),
            'hypervolume':          hv.calculate(archive),
            'generational_distance': gd.calculate(archive),
            'epsilon_indicator':    ei.calculate(archive),
            'inverted_gd':          ig.calculate(archive),
            'spacing':              sm.calculate(archive),
        })
    return pd.DataFrame(metrics).sort_values('nfe').reset_index(drop=True)


def plot_metrics(metrics, convergence, title=''):
    """Plot all 6 convergence indicators (5 archive-based + epsilon progress)."""
    sns.set_style('white')
    fig, axes = plt.subplots(nrows=6, figsize=(8, 12), sharex=True)
    ax1, ax2, ax3, ax4, ax5, ax6 = axes

    ax1.plot(metrics['nfe'], metrics['hypervolume'])
    ax1.set_ylabel('Hypervolume')

    ax2.plot(convergence['nfe'], convergence['epsilon_progress'])
    ax2.set_ylabel(r'$\varepsilon$-progress')

    ax3.plot(metrics['nfe'], metrics['generational_distance'])
    ax3.set_ylabel('Generational distance')

    ax4.plot(metrics['nfe'], metrics['epsilon_indicator'])
    ax4.set_ylabel(r'$\varepsilon$-indicator')

    ax5.plot(metrics['nfe'], metrics['inverted_gd'])
    ax5.set_ylabel('Inverted GD')

    ax6.plot(metrics['nfe'], metrics['spacing'])
    ax6.set_ylabel('Spacing')
    ax6.set_xlabel('Number of function evaluations')

    sns.despine(fig)
    if title:
        fig.suptitle(title, fontsize=12, y=1.01)
    plt.tight_layout()
    return fig

## 2. Load Lab 4 archives and compute convergence metrics

We load the archive files written by `ArchiveLogger` in Lab 4 and compute the full set
of convergence metrics. The reference set is the final archive (the best-known set for
this single run).

In [ ]:
# --- 5 000 NFE run ---
archives_5k = load_archives('./archives/lab4_run1.tar.gz')
archives_5k.sort(key=lambda x: x[0])
reference_set_5k = archives_5k[-1][1]   # final archive DataFrame = reference

metrics_5k = calculate_metrics(archives_5k, reference_set_5k)

# Load ε-progress saved by Lab 4 (avoids re-running the full optimisation)
convergence_5k_ep = pd.read_csv('./archives/lab4_convergence_5k.csv')

plot_metrics(metrics_5k, convergence_5k_ep, title='Convergence metrics — 5 000 NFE')
plt.show()


In [ ]:
# --- 100 000 NFE run ---
archives_100k = load_archives('./archives/lab4_run2.tar.gz')
archives_100k.sort(key=lambda x: x[0])
reference_set_100k = archives_100k[-1][1]

metrics_100k = calculate_metrics(archives_100k, reference_set_100k)

# Load ε-progress saved by Lab 4 (avoids re-running the full optimisation)
convergence_100k_ep = pd.read_csv('./archives/lab4_convergence_100k.csv')

plot_metrics(metrics_100k, convergence_100k_ep, title='Convergence metrics — 100 000 NFE')
plt.show()


**Interpretation:**
- At 5 000 NFE: hypervolume and ε-progress are still rising — not converged.
- At 100 000 NFE: hypervolume begins to plateau — improved but may need more NFE.
- GD, ε-indicator, and IGD should decrease toward zero as the archive approaches the reference set.

## 3. Pareto front visualisation — parallel coordinates

We now visualise the 100k Pareto front using parallel coordinates. Each line is one
non-dominated solution. `max_P` is inverted so that "up" means "better" for all axes.

Crossing lines between axes reveal trade-offs: solutions that are good on one objective
tend to be poor on another.

In [ ]:
obj_cols = ['max_P', 'utility', 'inertia', 'reliability']

# Load final archive from 100k run
archives_100k = load_archives('./archives/lab4_run2.tar.gz')
archives_100k.sort(key=lambda x: x[0])
results_pareto = archives_100k[-1][1]   # already a DataFrame with objective columns

fixed_limits = pd.DataFrame([[0, 0, 0, 0], [5, 2, 1, 1]], columns=obj_cols)
pc_axes = parcoords.ParallelAxes(fixed_limits)
pc_axes.plot(results_pareto[obj_cols])
pc_axes.invert_axis('max_P')
plt.title('Pareto front — 100 000 NFE, parallel coordinates')
plt.show()


## 4. The role of stochasticity — seed analysis

MOEAs use random number generators for crossover and mutation. Running the same algorithm
twice with different seeds will produce different (but hopefully similar) Pareto fronts.

**Best practice:** run 5 seeds, then combine the final archives into a cross-seed reference set
using ε-nondominated sorting. The shared reference set provides a fair basis for comparing
convergence across seeds.

In [ ]:
import os
results_seeds    = []
convergences_seeds = []

os.makedirs('./archives', exist_ok=True)
with MultiprocessingEvaluator(lake_model) as evaluator:
    for i in range(5):
        _sf = f'./archives/seed_{i}.tar.gz'
        if os.path.exists(_sf):
            os.remove(_sf)
        result, convergence = evaluator.optimize(
            nfe=5000,
            searchover='levers',
            epsilons=[0.125, 0.05, 0.01, 0.01],
            constraints=[constraint_maxP],
            filename=f'seed_{i}.tar.gz',
            directory='./archives',
        )
        results_seeds.append(result)
        convergences_seeds.append(convergence)

print(f'Seeds completed. Solutions per seed: {[len(r) for r in results_seeds]}')


### 4.1 Visualise Pareto fronts across seeds

Each seed produces a different Pareto front. We overlay them on the same parallel
coordinates plot using different colours. Tight clustering indicates good reproducibility;
wide spread suggests more NFE are needed.

In [ ]:
fixed_limits = pd.DataFrame([[0, 0, 0, 0], [5, 2, 1, 1]],
                             columns=['max_P', 'utility', 'reliability', 'inertia'])
pc_seeds = parcoords.ParallelAxes(fixed_limits)
palette  = sns.color_palette(n_colors=5)

for i, (result, color) in enumerate(zip(results_seeds, palette)):
    obj = result.loc[:, ['max_P', 'utility', 'reliability', 'inertia']]
    pc_seeds.plot(obj, color=color, label=f'Seed {i}')

pc_seeds.invert_axis('max_P')
pc_seeds.legend()
plt.title('Pareto fronts across 5 random seeds (5 000 NFE each)')
plt.show()

### 4.2 Cross-seed reference set

The reference set for seed analysis must be the **union of all seeds' final archives**,
filtered by ε-nondominated sorting. Using the final archive of a single seed as the reference
would bias the metrics toward that seed.

In [ ]:
problem = Problem('levers', lake_model.levers, [o for o in lake_model.outcomes if o.kind != o.INFO])

# Combine all seeds' final results and compute ε-nondominated reference set
reference_set_seeds = epsilon_nondominated(
    results_seeds,
    epsilons=[0.125, 0.05, 0.01, 0.01],
    problem=problem,
)
print(f'Cross-seed reference set: {len(reference_set_seeds)} solutions')

### 4.3 Per-seed convergence metrics

We compute all convergence metrics for each seed, using the shared cross-seed reference set.
Consistent curves across seeds (same colour trajectories converging to similar values)
indicate robust convergence.

In [ ]:
# Load per-seed archives
all_seed_archives = []
for i in range(5):
    arch = load_archives(f'./archives/seed_{i}.tar.gz')
    all_seed_archives.append(arch)

# Compute metrics per seed against the cross-seed reference set
metrics_by_seed = [calculate_metrics(arch, reference_set_seeds)
                   for arch in all_seed_archives]


In [ ]:
sns.set_style('white')
fig, axes = plt.subplots(nrows=6, figsize=(8, 12), sharex=True)
ax1, ax2, ax3, ax4, ax5, ax6 = axes

for metrics, convergence, color in zip(metrics_by_seed, convergences_seeds,
                                       sns.color_palette(n_colors=5)):
    ax1.plot(metrics['nfe'], metrics['hypervolume'], color=color)
    ax2.plot(convergence['nfe'], convergence['epsilon_progress'], color=color)
    ax3.plot(metrics['nfe'], metrics['generational_distance'], color=color)
    ax4.plot(metrics['nfe'], metrics['epsilon_indicator'], color=color)
    ax5.plot(metrics['nfe'], metrics['inverted_gd'], color=color)
    ax6.plot(metrics['nfe'], metrics['spacing'], color=color)

for ax, label in zip(axes, ['Hypervolume', r'$\varepsilon$-progress',
                              'Generational distance', r'$\varepsilon$-indicator',
                              'Inverted GD', 'Spacing']):
    ax.set_ylabel(label)

ax6.set_xlabel('Number of function evaluations')
fig.suptitle('Convergence metrics — 5 seeds (colours), cross-seed reference set',
             fontsize=12)
sns.despine(fig)
plt.tight_layout()
plt.show()

## Summary

You have computed the full suite of MOEA convergence metrics and interpreted them:

- **Hypervolume / ε-progress** as primary convergence indicators.
- **GD, ε-indicator, IGD** as distance-based proximity measures to the reference set.
- **Spacing** as a diversity/uniformity measure of the archive.

The seed analysis demonstrates that 5 000 NFE is insufficient for stable, reproducible
results. In practice, choose NFE where metrics plateau consistently across seeds.

In **Lab 4** you will use the optimised solutions as candidate policies for Multi-Objective
Robust Decision Making (MORDM) — re-evaluating them under deep uncertainty and computing
robustness metrics.